# Phase 3: Training the 3D Neural Radiance Field

In [ ]:
# Run this to mount your drive and cd into the HW4 folder automatically!
import sys
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')

    %cd "/content/drive/MyDrive/Study/Second Sem/CV2/HW/HW4"
    sys.path.append('/content/drive/MyDrive/Study/Second Sem/CV2/HW/HW4')
except:
    print("Not running on Google Colab, assuming local execution.")

In [ ]:
# Ensure our output directory exists for saving images and models so nothing is ever lost!
os.makedirs('output/phase3', exist_ok=True)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output

# Import our geometry math implemented previously!
from src.dataset_3d import load_data, RaysData
from src.rendering import sample_along_rays, volrend

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

### 1. The Model Architecture (`NeuralRadianceField`)
This matches the exact diagram from the TA instructions:

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, L=10):
        super().__init__()
        self.L = L

    def forward(self, x):
        pe = [x]
        for i in range(self.L):
            freq = 2.0 ** i
            # Removed `* torch.pi` from the two lines below to prevent gradient explosion!
            pe.append(torch.sin(freq * x))
            pe.append(torch.cos(freq * x))
        return torch.cat(pe, dim=-1)


class NeuralRadianceField(nn.Module):
    def __init__(self, L_xyz=10, L_dir=4, d_filter=256):
        super().__init__()
        self.pe_xyz = PositionalEncoding(L_xyz)
        self.pe_dir = PositionalEncoding(L_dir)

        d_input = 3 + 3 * 2 * L_xyz
        d_direction = 3 + 3 * 2 * L_dir

        # Backbone (Layers 1-4)
        self.layer1 = nn.Linear(d_input, d_filter)
        self.layer2 = nn.Linear(d_filter, d_filter)
        self.layer3 = nn.Linear(d_filter, d_filter)
        self.layer4 = nn.Linear(d_filter, d_filter)

        # Backbone (Layers 5-8)
        # Skip Connection: We concat original xyzs info back into the 5th layer
        self.layer5 = nn.Linear(d_filter + d_input, d_filter)
        self.layer6 = nn.Linear(d_filter, d_filter)
        self.layer7 = nn.Linear(d_filter, d_filter)
        self.layer8 = nn.Linear(d_filter, d_filter) # Original NeRF drops the ReLU right before the split!

        # Density Branch
        self.sigma_out = nn.Linear(d_filter, 1)

        # Color (RGB) Branch, which looks at identical geometry + direction
        self.rgb_layer1 = nn.Linear(d_filter + d_direction, 128)
        self.rgb_out = nn.Linear(128, 3)

        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, xyzs, r_ds):
        # xyzs: (num_rays, num_samples, 3)
        # r_ds: (num_rays, 3)
        num_rays, num_samples, _ = xyzs.shape

        # 1. Encode spatial and directional points
        x_enc = self.pe_xyz(xyzs) # (num_rays, num_samples, 63)
        d_enc = self.pe_dir(r_ds) # (num_rays, 27)

        # Expand the single ray direction to cover all samples hitting that direction
        d_enc = d_enc.unsqueeze(1).expand(-1, num_samples, -1) # -> (num_rays, num_samples, 27)

        # 2. Main Neural Network pass
        h = self.relu(self.layer1(x_enc))
        h = self.relu(self.layer2(h))
        h = self.relu(self.layer3(h))
        h = self.relu(self.layer4(h))

        # Skip connection!
        h = torch.cat([h, x_enc], dim=-1)
        h = self.relu(self.layer5(h))
        h = self.relu(self.layer6(h))
        h = self.relu(self.layer7(h))
        h = self.layer8(h)

        # 3. Density prediction -> Output MUST be entirely independent of View Direction!
        sigma = self.relu(self.sigma_out(h))

        # 4. Color prediction -> View Direction is concatenated here!
        h_color = torch.cat([h, d_enc], dim=-1)
        rgb = self.relu(self.rgb_layer1(h_color))
        rgb = self.sigmoid(self.rgb_out(rgb))

        return rgb, sigma

### 2. Loading the Data

In [ ]:
# Load Pre-calculated datasets from Phase 2
data_path = "data/lego_200x200.npz"
images_train, c2ws_train, images_val, c2ws_val, c2ws_test, K = load_data(data_path=data_path)

# Ensure tensors for geometry calculations
if isinstance(images_train, np.ndarray): images_train = torch.from_numpy(images_train).float()
if isinstance(c2ws_train, np.ndarray): c2ws_train = torch.from_numpy(c2ws_train).float()
if isinstance(images_val, np.ndarray): images_val = torch.from_numpy(images_val).float()
if isinstance(c2ws_val, np.ndarray): c2ws_val = torch.from_numpy(c2ws_val).float()
if isinstance(K, np.ndarray): K = torch.from_numpy(K).float()

# Instantiate the massive Ray Datasets pool
dataset_train = RaysData(images_train.to(device), K.to(device), c2ws_train.to(device), device=device)
dataset_val = RaysData(images_val.to(device), K.to(device), c2ws_val.to(device), device=device)

model = NeuralRadianceField().to(device)
optimizer = optim.Adam(model.parameters(), lr=5e-4)
criterion = nn.MSELoss()

### 3. Validation Logic (Abstracted for readability)
Because rendering 6 images takes heavily chunked GPU cycles, we put it into a helper function.

In [ ]:
def evaluate_and_plot(model, iteration, loss_history, psnr_val_history, dataset_val, near, far, num_samples, chunk_size=16384):
    model.eval()
    H, W = dataset_val.h, dataset_val.w
    num_val_images = dataset_val.num_images

    # Get global arrays of exactly all validation data
    rays_o_val_all = dataset_val.rays_o.reshape(num_val_images, -1, 3)
    rays_d_val_all = dataset_val.rays_d.reshape(num_val_images, -1, 3)
    gt_img_val_all = dataset_val.gt_rgbs.reshape(num_val_images, -1, 3)

    with torch.no_grad():
        val_psnrs = []
        rendered_val_first = None

        for val_idx in range(num_val_images):
            rays_o_val = rays_o_val_all[val_idx]
            rays_d_val = rays_d_val_all[val_idx]
            gt_img_val = gt_img_val_all[val_idx]
            rendered_val = []

            # Chunking so the GPU doesn't crash on the 40k validation rays
            for chunk_idx in range(0, rays_o_val.shape[0], chunk_size):
                ro_batch = rays_o_val[chunk_idx:chunk_idx+chunk_size]
                rd_batch = rays_d_val[chunk_idx:chunk_idx+chunk_size]

                xyzs_val = sample_along_rays(ro_batch, rd_batch, near, far, num_samples, perturb=False, device=device)
                rgb_b, sig_b = model(xyzs_val, rd_batch)
                chunk_rendered = volrend(sig_b, rgb_b, near, far, num_samples, device=device)

                rendered_val.append(chunk_rendered)

            rendered_val = torch.cat(rendered_val, dim=0)
            val_mse = criterion(rendered_val, gt_img_val)
            val_psnr = 10 * torch.log10(1 / val_mse).item()
            val_psnrs.append(val_psnr)

            # Only convert the first image for the visual plot matrix
            if val_idx == 0:
                rendered_val_first = rendered_val.reshape(H, W, 3).cpu().numpy().clip(0, 1)

        avg_psnr = np.mean(val_psnrs)
        psnr_val_history.append(avg_psnr)

        # HARD SAVE 1: Save the actual iteration image
        plt.imsave(f'output/phase3/val_step_{iteration}.png', rendered_val_first)

        # HARD SAVE 2: Save the raw numpy arrays for loss and PSNR so you never lose your math data!
        np.save('output/phase3/loss_history.npy', np.array(loss_history))
        np.save('output/phase3/psnr_val_history.npy', np.array(psnr_val_history))

        clear_output(wait=True)
        plt.figure(figsize=(15, 4))

        plt.subplot(1, 3, 1)
        plt.imshow(rendered_val_first)
        plt.title(f"Validation View 0 (Iteration {iteration})\nImage PSNR: {val_psnrs[0]:.2f} dB")
        plt.axis('off')

        plt.subplot(1, 3, 2)
        plt.plot(loss_history)
        plt.title(f"Training Loss Log")
        plt.yscale('log')
        plt.grid(True)

        plt.subplot(1, 3, 3)
        psnr_steps = [s * 500 for s in range(len(psnr_val_history))]
        plt.plot(psnr_steps, psnr_val_history, color='orange')
        plt.title(f"Avg 6-Image Val PSNR: {avg_psnr:.2f} dB")
        plt.grid(True)

        # HARD SAVE 3: Save the graph figure perfectly
        plt.savefig(f'output/phase3/training_curves.png')
        plt.show()

        return avg_psnr

### 4. The Training Loop
Because we moved the validation out, the raw backpropagation loop is much cleaner.

In [ ]:
# Hyperparameters
num_iterations = 6000
batch_size = 10000
near = 2.0
far = 6.0
num_samples = 64

loss_history = []
psnr_val_history = []

print("Training Started! Outputting checkpoints to output/phase3...\n")

for i in range(num_iterations + 1):
    model.train()

    # 1. Grab random rays
    rays_o, rays_d, target_rgb = dataset_train.sample_rays(batch_size)

    # 2. Slice rays into 64 dots between `near` and `far` (Jittering activated!)
    xyzs = sample_along_rays(rays_o, rays_d, near, far, num_samples, perturb=True, device=device)

    # 3. Model predicts Color and Density (sigma)
    rgbs, sigmas = model(xyzs, rays_d)

    # 4. Integrate/squash dots into one solid color
    pred_rgb = volrend(sigmas, rgbs, near, far, num_samples, device=device)

    # 5. Backprop
    optimizer.zero_grad()
    loss = criterion(pred_rgb, target_rgb)
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

    if i % 25 == 0:
      print(f"Iteration {i}/{num_iterations} - Loss: {loss.item():.4f}")


    # VISUALIZATION CHECKPOINT: Every 500 steps, call our clean helper function!
    if i % 500 == 0 or i == num_iterations:
        evaluate_and_plot(model, i, loss_history, psnr_val_history, dataset_val, near, far, num_samples)

# HARD SAVE 4: Unconditionally permanently save the actual model PyTorch weights!
torch.save(model.state_dict(), "output/phase3/nerf_model_weights.pth")
print("Training Complete! Validations, Graphics, History, and Model Weights permanently saved to output/phase3/")